## ragas

🤦‍♂️需要调用llm，免费不够，搞不了。

不需要手动生成测试集，让LLM读文档，用ai练ai。ragas自带了`TestsetGenerator `

考官：是更强的。不一样的llm,不一样的向量嵌入。  我们要确保考官更加聪明

In [1]:
from ragas.testset import TestsetGenerator
import os
from dotenv import load_dotenv

load_dotenv()

/home/dong/miniconda3/envs/data-analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(os.path.join(os.getcwd(), 'data'), glob="*.md")
docs = loader.load()

No features in text.
short text: "liangfen". Defaulting to English.
short text: "1". Defaulting to English.
short text: "bz2". Defaulting to English.
short text: "bz4". Defaulting to English.
short text: "bz5". Defaulting to English.
short text: "bz6". Defaulting to English.
short text: "bz7". Defaulting to English.
short text: "bz7". Defaulting to English.
short text: "bz7". Defaulting to English.
short text: "bz7". Defaulting to English.
short text: "bz7". Defaulting to English.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
short text: "citrus-tea". Defaulting to English.
No features in text.
No features in text.
short text: "tea". Defaulting to English.
short text: "tea". Defaulting to English.
short text: "image". Defaulting to English.
short text: "image". Defaulting to English.
No features in text.
No features in text.

In [3]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
# llm改用gemini
llm = ChatOpenAI(
    api_key= os.getenv('AIHUBMIX_API_KEY_GEMINI'),
    base_url="https://aihubmix.com/v1",
    model='gemini-3-flash-preview-free',
) 
generator_llm = LangchainLLMWrapper(llm)

# 使用同样的向量嵌入模型
generator_embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-zh-v1.5',
)

/tmp/ipykernel_18868/3289352527.py:11: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(llm)


In [4]:
# 用户画像
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="家庭主妇",
        role_description="一个退休的家庭主妇，想要学习如何做菜",
    ),
]

In [5]:
# 
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings, persona_list=personas
)

In [6]:
# 文本分块提取。
from ragas.testset.transforms.extractors.llm_based import NERExtractor,HeadlinesExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter

transforms = [
    HeadlinesExtractor(llm=generator_llm), # 语义上标注，在metadata中标记 header
    HeadlineSplitter(), # 按照header切割文本 ，物理剪切
    NERExtractor(llm=generator_llm) # 语义标注， 使用llm提取文本中的命名实体，。llm这里为了方便就选了一样的
    ]

In [7]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer, 
)

# 配置题目类型比例
# singhop 单跳查询：在一个文档查询就可以，不跨文档
# 
distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0), 
]
# Synthesizer有出题的内部规则，有prompt模板，是英文的，我们需要把他转换为中文
for query, _ in distribution:
    prompts = await query.adapt_prompts("chinese", llm=generator_llm)
    query.set_prompts(**prompts)

RateLimitError: Error code: 429 - {'error': {'message': 'Sorry, you have reached the limit of the free model quota. Please switch to a paid model to enjoy unlimited concurrency. https://console.aihubmix.com/topup (tid: 2026050302531828542206061495648)', 'type': 'Aihubmix_api_error'}}

比如有实体，红烧肉，  
- 查询器启动，将文档和红烧肉 给llm，
- 命令llm：“根据这个实体和原文，给我出一个具体的、单步就能回答的问题。”
- llm生成问题和答案。

adapt_prompts就是把 命令LLM通常是英文规则，叫 'Generate a question'，翻译成中文命令

In [ ]:
len(docs)

In [ ]:
from ragas.run_config import RunConfig
dataset = generator.generate_with_langchain_docs(
    docs[:1],
    testset_size=5,
    transforms=transforms,
    query_distribution=distribution,
)

🤦‍♂️🤦‍♂️
- ragas是并发的，
- 很多地方都使用了llm调用，很容易就超过免费模型调用速率限制。

本地内存太小，部署不了。 

付费是为了算力付费，大模型本身是一个复杂函数，没什么。
